# Import GAS history

**Can this register start from the history the Apps Script app already has, instead of from today?**

The only notebook here that is meant to be run once. It seeds the ledger and the scan log from
a migration bundle exported by the GAS **Data** page, so the first real scan continues those
lifecycles rather than beginning new ones.

## Why this exists, and when not to run it

A ledger that starts empty is not merely incomplete, it is wrong: every `first_seen` collapses
to the moment of the first scan, so Kaplan–Meier reads near zero, the capacity grid marks every
earlier month `reconstructed`, and coverage and efficiency are computed over a population one
scan deep. None of that looks like an error.

Run this **before** the first pipeline run, once. It refuses a register that holds anything —
the ledger, the scan log, or any of the six appended tables — because merging a seed into a
register that has already scanned would re-open lifecycles it has since resolved and hand the
disappearance guard the wrong previous scan.

`force_import` **replaces the register**: it overwrites the ledger and the scan log and empties
bronze, silver and the four gold tables. The gold tables are why it goes that far — they are
computed from the ledger as it stood at each scan, so rows written before a seed came from a
ledger that started empty, and left in place they read as a near-zero-MTTR run sitting beside
seeded ones with nothing on the page to explain the gap. Re-scan to repopulate them.

A **"No write access"** failure here is Unity Catalog, not the bundle: it is checked before any
of the expensive work so the message names the grant. Overwriting is not a way around it — UC
gives a table's owner `MODIFY` implicitly, so being refused it means this principal does not own
the table, and replacing or dropping needs ownership or `MANAGE`.

Two things must line up, and getting either wrong invents remediation that never happened:

- **`--severities` on the first scan must match what GAS was scanning.** Absence of a severity
  nobody looked for is not a fix.
- **`--project_id` must match GAS's `WIZ_PROJECT_ID_V2`.** GAS scans one Wiz project; brick's
  `os` scope pins none unless asked. If brick's population is wider or narrower, the first run
  resolves-by-disappearance everything outside the overlap.

What does **not** come across, stated once so it is not discovered later: `tags_json` (brick's
ingest selects no asset tags), the actionable clock (unimplemented here), and per-scan bronze —
the bundle carries reconciled lifecycles, not raw findings, so the gold tables begin
accumulating from the first brick run.

In [ ]:
import os, sys

_paths = []
try:
    _paths.append(dbutils.widgets.get("module_path"))
except Exception:  # noqa: BLE001 -- the widget does not exist yet on a first run
    pass
_here = os.getcwd()
_paths += [_here, os.path.dirname(_here)]
for _p in _paths:
    if _p and os.path.exists(os.path.join(_p, "run_pipeline.py")):
        sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("brick modules are not on sys.path -- see brick/README.md, step 2")

import import_bundle, run_pipeline

# Declared here rather than through panels.declare_widgets: this notebook writes and does not
# draw, so it has no business importing the chart layer. `catalog` has no default on purpose --
# these tables map unpatched CVEs to named hosts, and a default that succeeds is the wrong
# failure mode. See brick/README.md, "Store the credentials".
for _name, _default in (
    ("catalog", ""), ("schema", "industry"), ("scope", "os"), ("table_prefix", ""),
    ("module_path", ""), ("bundle_path", ""), ("force_import", "false"),
    ("data_path", ""),  # set for a PoC register with no catalog; see the README
):
    dbutils.widgets.text(_name, _default)

## Is the deployment sound?

Every module must report the same version **from the same folder**. `import_bundle` writes the
ledger, so a stale copy of it beside a fresh `ledger.py` is as fatal as a stale `metrics.py` —
it imports cleanly and then writes rows the reconciler cannot continue.

The printed path is the other half: `config`, `metrics` and `ingest` are generic module names.
If something else on `sys.path` shadows one, the path below says so.

In [ ]:
import config, dbx, ingest, ledger, metrics

for _m in (config, dbx, ingest, ledger, metrics, run_pipeline, import_bundle):
    print(f"{_m.__name__:14} {getattr(_m, 'MODULE_VERSION', 'PRE-2.0 — STALE'):8}"
          f" {_m.__file__}")

## Read the bundle

Export it from the GAS app: **Data → Migration bundle (Drive)**, then upload the
`migration-….json.gz` it hands you somewhere the cluster can read — a Unity Catalog volume is
the obvious place:

```
/Volumes/<catalog>/<schema>/<volume>/migration-20260811T000000Z.json.gz
```

This cell only reads and validates. Nothing is written until the cell after it, so the counts
below are the last chance to notice that the bundle is from the wrong deployment.

In [ ]:
_bundle = import_bundle.load_bundle(run_pipeline.param("bundle_path"))
_episodes, _collapsed = import_bundle.selectable_episodes(_bundle)

print(f"exported_at   {_bundle.get('exported_at')}")
print(f"scans         {len(_bundle['scans'])}")
print(f"lifecycles    {len(_bundle['ledger'])}")
print(f"episodes      {len(_bundle['episodes'])} ({len(_episodes)} will be folded in)")
print(f"mttr_history  {len(_bundle['mttr_history'])}  (read by nothing here -- see the README)")

## Seed the ledger and the scan log

Two tables, written together. They have to be: `reconcile`'s disappearance branch fires only
when a row's `last_scan_id` names the immediately-previous scan in the log, so a ledger seeded
without its scans would freeze — nothing would ever resolve — and a scan log without its ledger
would resolve everything at once.

In [ ]:
_namespace = run_pipeline.resolve_namespace()
_scope = run_pipeline.resolve_scope()
_tables = run_pipeline.resolve_tables(_namespace, _scope)

run_pipeline.ensure_schema(spark, _namespace)
_summary = import_bundle.import_bundle(
    spark, _tables, _bundle, scope=_scope,
    force=run_pipeline.truthy(run_pipeline.param("force_import")),
)
import_bundle.summarize(_summary, _tables)

## Did it take?

`earliest_first_seen` is the number to read. If it says today, the seed did not land and every
MTTR figure below it is measuring this import rather than the register.

`kev` and `epss_captured` are the other half: coverage and efficiency classify from the exploit
signals, and a NULL there means *never captured*, not *not exploitable*. A seed where almost
nothing is captured will publish honest-but-wide bounds rather than a confident wrong rate — run
GAS's **Settings → Risk-signal backfill** first and export again if that is what you see.

In [ ]:
display(import_bundle.seeded_overview(spark, _tables))

## Next

Open **`06_run_and_verify`** and run one scan, with `severities` set to what GAS was scanning
and `project_id` set to its `WIZ_PROJECT_ID_V2`.

Read `resolved_count` in that run's summary before anything else. A plausible day's remediation
means the handoff worked. A number close to the whole register means the two populations
disagree — re-import with corrected parameters rather than accepting the result, because after
the second run the mistake is indistinguishable from a real mass closure.